# Wan2.1 VACE 14B — H3 カット生成

既存の `wan22_i2v_lightx2v_colab.ipynb` と同じく、モデルは Google Drive に一度置いて再利用する。有料 API は使わない。

ランタイムは **A100 以上**。最初のコードセルで、それより小さい GPU は停止する。

上から順に実行する。起動手順とライセンスは同じフォルダの `README.md`。

1. GPU チェック
2. 制御 JSON の読み込みと、OpenPose / Depth / カメラ制御の作成
3. Drive をマウントし、ComfyUI を入れる
4. モデル取得（Drive にあればスキップ）
5. ComfyUI 起動
6. カットごとに 9:16 の mp4 と、使った制御 JSON を書く
7. 生成が終わったらランタイムを切断して削除する

入力は `マイドライブ/ComfyUI_vace_h3/input/face_ref.png` と `input/controls/*.json`。
出力は `マイドライブ/ComfyUI_vace_h3/output/<実行時刻>/`。

このノートを置いた時点では、A100 上の生成は未実施。


In [ ]:
import subprocess

A100_OR_BETTER = ("A100", "H100", "H200", "B100", "B200", "GH200")
MIN_VRAM_MIB = 38000


def gpu_is_a100_or_better(name, vram_mib):
    upper = str(name).upper()
    named = any(token in upper for token in A100_OR_BETTER)
    return named and int(vram_mib) >= MIN_VRAM_MIB


def gpu_stop_message(name, vram_mib):
    return (
        "このノートは A100 以上の GPU が必要です。小さい GPU ではここで止めます。生成は開始しません。\n"
        f"検出した GPU: {name} / {vram_mib} MiB\n"
        "ランタイム → ランタイムのタイプを変更 → A100 を選び、ランタイムを接続し直してから、最初のセルを再実行してください。\n"
        "T4 / L4 / V100 / A10 では動きません。"
    )


def parse_nvidia_smi(text):
    rows = []
    for line in str(text).splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) < 2 or not parts[0]:
            continue
        mem = parts[1].lower().replace("mib", "").strip()
        try:
            vram = int(float(mem))
        except ValueError:
            continue
        rows.append((parts[0], vram))
    return rows


def evaluate_gpus(rows):
    if not rows:
        return False, "(なし)", 0, gpu_stop_message("(なし)", 0)
    good = [row for row in rows if gpu_is_a100_or_better(row[0], row[1])]
    if good:
        name, vram = max(good, key=lambda row: row[1])
        return True, name, vram, ""
    name, vram = max(rows, key=lambda row: row[1])
    return False, name, vram, gpu_stop_message(name, vram)


def assert_a100_or_stop():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0 or not result.stdout.strip():
        message = gpu_stop_message("(nvidia-smi 失敗)", 0)
        print(message)
        if result.stderr.strip():
            print(result.stderr.strip())
        raise RuntimeError(message)
    ok, name, vram, message = evaluate_gpus(parse_nvidia_smi(result.stdout))
    if not ok:
        print(message)
        raise RuntimeError(message)
    print(f"GPU OK: {name} / {vram} MiB（A100 以上）")
    return name, vram


_GPU_NAME, _GPU_VRAM = assert_a100_or_stop()
GPU_OK = True


In [ ]:
"""Wan2.1 VACE 14B Colab が、カット制御 JSON をそのまま読むための関数。

JSON のキーは書き換えない。描画に使う単位は次のとおり。

- camera_vector.translate: ピクセル。x は右、y は下、z はドリー（フレーム高さに対する倍率）。
- camera_vector.rotate: 度。x はピッチ、y はヨー、z はロール。
- camera_vector.shake.amp_px: 手ブレの振幅（ピクセル）。seed で決まる。
- wrist_to_handle_cm: 1cm をフレーム高さの 2% として制御画像に描く。
- OpenPose / Depth の学習済み重みは使わない。線と濃淡はこのファイルが描く。
- focus_sweep と openpose の時刻範囲の外は、端の値を保持する。
- VACE のフレーム数は 4n+1、最大 81。duration_s * fps に最も近い本数へ揃える。
"""

import json
import math
import random
import re
from pathlib import Path

from PIL import Image, ImageDraw, ImageFilter

FACE_FILENAMES = ("face_ref.png", "face_ref.jpg", "face_ref.jpeg", "face_ref.webp")
EXPORT_WIDTH = 720
EXPORT_HEIGHT = 1280
GEN_480P = (480, 832)
GEN_720P = (720, 1280)
VRAM_NATIVE_9_16_MIB = 70000
MAX_VACE_FRAMES = 81
CM_OF_FRAME_HEIGHT = 0.02
MIN_GAP_CM = 1.0
HANDLE_XY = (0.68, 0.50)
CAT_XY = (0.38, 0.50)
_WRIST_X = 0.45
_WRIST_Y = 0.75
_WRIST_NORM = math.hypot(_WRIST_X, _WRIST_Y)
WRIST_DIR = (_WRIST_X / _WRIST_NORM, _WRIST_Y / _WRIST_NORM)
FACE_PAD = (128, 128, 128)

POSES = {
    "back_to_camera": {"facing": 0.0, "paw": 0.0},
    "face_to_camera": {"facing": 1.0, "paw": 0.0},
    "front": {"facing": 1.0, "paw": 0.0},
    "forepaw_to_mouth": {"facing": 1.0, "paw": 1.0},
    "paw_returned": {"facing": 1.0, "paw": 0.0},
}

DEPTH_LOOK = {
    "near": {"bg": 25, "cat": 235, "cat_scale": 1.20, "handle": 70},
    "far": {"bg": 110, "cat": 140, "cat_scale": 0.62, "handle": 120},
    "cat": {"bg": 30, "cat": 240, "cat_scale": 1.05, "handle": 45},
    "handle": {"bg": 28, "cat": 60, "cat_scale": 0.85, "handle": 245},
}

_BACK = {
    "head": (0.38, 0.26),
    "lear": (0.31, 0.20),
    "rear": (0.45, 0.20),
    "neck": (0.38, 0.36),
    "chest": (0.38, 0.48),
    "lsho": (0.30, 0.46),
    "rsho": (0.46, 0.46),
    "pelvis": (0.38, 0.64),
    "lhind": (0.30, 0.86),
    "rhind": (0.46, 0.86),
    "tail": (0.22, 0.62),
}
_FRONT = {
    "head": (0.38, 0.26),
    "lear": (0.30, 0.18),
    "rear": (0.46, 0.18),
    "neck": (0.38, 0.36),
    "chest": (0.38, 0.48),
    "lsho": (0.26, 0.46),
    "rsho": (0.50, 0.46),
    "pelvis": (0.38, 0.64),
    "lhind": (0.28, 0.88),
    "rhind": (0.48, 0.88),
    "tail": (0.56, 0.60),
}
_LF_REST = (0.24, 0.72)
_RF_REST = (0.52, 0.72)
_LF_MOUTH = (0.36, 0.34)
_LIMBS = (
    ("lear", "head", (255, 0, 85)),
    ("rear", "head", (255, 0, 170)),
    ("head", "neck", (255, 0, 0)),
    ("neck", "chest", (255, 85, 0)),
    ("chest", "lsho", (85, 255, 0)),
    ("lsho", "lf_paw", (0, 255, 85)),
    ("chest", "rsho", (255, 170, 0)),
    ("rsho", "rf_paw", (170, 255, 0)),
    ("chest", "pelvis", (0, 255, 255)),
    ("pelvis", "lhind", (0, 85, 255)),
    ("pelvis", "rhind", (0, 0, 255)),
    ("pelvis", "tail", (170, 0, 255)),
)


def safe_cut_id(cut_id):
    if not re.fullmatch(r"[A-Za-z]+[0-9]+", str(cut_id)):
        raise ValueError(f"cut 名が不正です: {cut_id}")
    return str(cut_id)


def cut_sort_key(cut_id):
    match = re.fullmatch(r"([A-Za-z]+)([0-9]+)", str(cut_id))
    if match is None:
        return (1, 0, str(cut_id))
    return (0, int(match.group(2)), match.group(1))


def _vec3(value, label):
    if not isinstance(value, (list, tuple)) or len(value) != 3:
        raise ValueError(f"camera_vector.{label} は数値3つです")
    return [float(item) for item in value]


def _pose_from_name(name):
    if name not in POSES:
        known = ", ".join(sorted(POSES))
        raise ValueError(f"未知の openpose ラベルです: {name}。既知: {known}")
    return POSES[name]


def _depth_look(label):
    if label not in DEPTH_LOOK:
        known = ", ".join(sorted(DEPTH_LOOK))
        raise ValueError(f"未知の depth ラベルです: {label}。既知: {known}")
    return DEPTH_LOOK[label]


def load_cut(path):
    path = Path(path)
    raw_text = path.read_text(encoding="utf-8")
    data = json.loads(raw_text)
    if not isinstance(data, dict):
        raise ValueError(f"{path}: 制御 JSON はオブジェクトです")
    required = (
        "cut",
        "source_seconds",
        "aspect",
        "fps",
        "openpose_keyframes",
        "depth",
        "camera_vector",
        "duration_s",
    )
    missing = [key for key in required if key not in data]
    if missing:
        raise ValueError(f"{path}: 足りないキー: {', '.join(missing)}")
    cut_id = safe_cut_id(data["cut"])
    if data.get("aspect") != "9:16":
        raise ValueError(f"{path}: aspect は 9:16 だけを受けます: {data.get('aspect')}")
    duration_s = float(data["duration_s"])
    fps = float(data["fps"])
    if duration_s <= 0 or fps <= 0:
        raise ValueError(f"{path}: duration_s と fps は正の数です")
    source = data["source_seconds"]
    if not isinstance(source, (list, tuple)) or len(source) != 2:
        raise ValueError(f"{path}: source_seconds は [開始, 終了] です")
    t0 = float(source[0])
    t1 = float(source[1])
    if t1 <= t0:
        raise ValueError(f"{path}: source_seconds の終了は開始より後です")
    if abs((t1 - t0) - duration_s) > 0.051:
        raise ValueError(
            f"{path}: source_seconds の長さ {t1 - t0} と duration_s {duration_s} が一致しません"
        )
    keyframes = data["openpose_keyframes"]
    if not isinstance(keyframes, list) or not keyframes:
        raise ValueError(f"{path}: openpose_keyframes が空です")
    for kf in keyframes:
        if not isinstance(kf, dict) or "t" not in kf:
            raise ValueError(f"{path}: openpose キーフレームに t がありません")
        float(kf["t"])
        if "cat" in kf:
            _pose_from_name(kf["cat"])
        if "cat_jaw_deg" in kf:
            float(kf["cat_jaw_deg"])
        if "wrist_to_handle_cm" in kf:
            float(kf["wrist_to_handle_cm"])
    depth = data["depth"]
    if not isinstance(depth, dict) or "focus_sweep" not in depth:
        raise ValueError(f"{path}: depth.focus_sweep がありません")
    sweep = depth["focus_sweep"]
    if not isinstance(sweep, list) or not sweep:
        raise ValueError(f"{path}: depth.focus_sweep が空です")
    for item in sweep:
        if not isinstance(item, (list, tuple)) or len(item) != 2:
            raise ValueError(f"{path}: focus_sweep の要素は [時刻, ラベル] です")
        float(item[0])
        if not isinstance(item[1], str) or not item[1]:
            raise ValueError(f"{path}: depth ラベルが文字列ではありません")
        _depth_look(item[1])
    if "sakura_hands" in depth and depth["sakura_hands"] != "defocused":
        raise ValueError(f"{path}: sakura_hands は defocused だけを受けます")
    camera = data["camera_vector"]
    if not isinstance(camera, dict):
        raise ValueError(f"{path}: camera_vector がオブジェクトではありません")
    _vec3(camera.get("translate"), "translate")
    _vec3(camera.get("rotate"), "rotate")
    if "shake" in camera and camera["shake"] is not None:
        shake = camera["shake"]
        if not isinstance(shake, dict):
            raise ValueError(f"{path}: shake がオブジェクトではありません")
        if shake.get("type") != "handheld":
            raise ValueError(f"{path}: 未対応の shake.type です: {shake.get('type')}")
        amp = shake.get("amp_px")
        if not isinstance(amp, (list, tuple)) or not 1 <= len(amp) <= 2:
            raise ValueError(f"{path}: shake.amp_px は数値1つか2つです")
        float(amp[0])
        if len(amp) == 2:
            float(amp[1])
        if "seed" not in shake:
            raise ValueError(f"{path}: shake.seed がありません")
        int(shake["seed"])
        if "irregular" in shake and not isinstance(shake["irregular"], bool):
            raise ValueError(f"{path}: shake.irregular は true / false です")
    if "caption" in data:
        if not isinstance(data["caption"], list):
            raise ValueError(f"{path}: caption は配列です")
        for cap in data["caption"]:
            if not isinstance(cap, dict) or "text" not in cap or "t" not in cap:
                raise ValueError(f"{path}: caption に t と text がありません")
            if not isinstance(cap["t"], (list, tuple)) or len(cap["t"]) != 2:
                raise ValueError(f"{path}: caption.t は [開始, 終了] です")
    if "jaw_max_deg" in data:
        limit = float(data["jaw_max_deg"])
        for kf in keyframes:
            if "cat_jaw_deg" in kf and float(kf["cat_jaw_deg"]) - limit > 1e-6:
                raise ValueError(f"{path}: cat_jaw_deg が jaw_max_deg を超えています")
    if "sakura" in data and data["sakura"] != "static":
        raise ValueError(f"{path}: 未対応の sakura です: {data['sakura']}")
    if "contact" in data and not isinstance(data["contact"], bool):
        raise ValueError(f"{path}: contact は true / false です")
    if "hard_cut_before_touch" in data and not isinstance(data["hard_cut_before_touch"], bool):
        raise ValueError(f"{path}: hard_cut_before_touch は true / false です")
    if "fingers_visible" in data:
        fingers = int(data["fingers_visible"])
        if fingers < 0 or fingers > 5:
            raise ValueError(f"{path}: fingers_visible は 0 から 5 です")
    return {
        "path": str(path),
        "raw_text": raw_text,
        "data": data,
        "cut": cut_id,
        "duration_s": duration_s,
        "fps": fps,
        "aspect": data["aspect"],
        "openpose_keyframes": keyframes,
        "depth": depth,
        "camera_vector": camera,
        "source_seconds": (t0, t1),
    }


def list_cuts(directory):
    directory = Path(directory)
    if not directory.is_dir():
        raise FileNotFoundError(f"制御 JSON のフォルダがありません: {directory}")
    paths = sorted(
        path for path in directory.iterdir() if path.is_file() and path.suffix.lower() == ".json"
    )
    if not paths:
        raise FileNotFoundError(
            f"制御 JSON がありません。{directory} に C1 から C4 の json を、中身を変えずに置いてください。"
        )
    cuts = [load_cut(path) for path in paths]
    seen = {}
    for cut in cuts:
        if cut["cut"] in seen:
            raise ValueError(f"cut {cut['cut']} が重複しています: {seen[cut['cut']]} と {cut['path']}")
        seen[cut["cut"]] = cut["path"]
    cuts.sort(key=lambda cut: cut_sort_key(cut["cut"]))
    return cuts


def find_face_reference(directory):
    directory = Path(directory)
    matches = [directory / name for name in FACE_FILENAMES if (directory / name).is_file()]
    if not matches:
        names = ", ".join(FACE_FILENAMES)
        raise FileNotFoundError(
            "顔の参照画像がありません。"
            f"顔だけの切り抜きを {directory} に置いてください。"
            f"ファイル名は {names} のいずれかです。"
        )
    return matches[0]


def copy_control_json(cut, dest_dir):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / Path(cut["path"]).name
    dest.write_bytes(Path(cut["path"]).read_bytes())
    return dest


def wan_frame_count(duration_s, fps, max_frames=MAX_VACE_FRAMES):
    raw = max(1, int(round(float(duration_s) * float(fps))))
    snapped = int(round((raw - 1) / 4.0)) * 4 + 1
    cap = int(max_frames)
    if cap < 1:
        raise ValueError("max_frames は 1 以上です")
    max_ok = ((cap - 1) // 4) * 4 + 1
    if snapped > max_ok:
        snapped = max_ok
    return max(1, snapped)


def raw_frame_count(duration_s, fps):
    return max(1, int(round(float(duration_s) * float(fps))))


def frame_source_time(cut, index, count):
    t0, t1 = cut["source_seconds"]
    if count <= 1:
        return float(t0)
    return float(t0) + (float(t1) - float(t0)) * (index / (count - 1))


def index_for_time(cut, t, count):
    t0, t1 = cut["source_seconds"]
    if count <= 1 or t1 == t0:
        return 0
    frac = (float(t) - float(t0)) / (float(t1) - float(t0))
    index = int(round(frac * (count - 1)))
    return max(0, min(count - 1, index))


def generation_geometry(vram_mib, override=None):
    if override is not None:
        width, height = int(override[0]), int(override[1])
        if width % 16 or height % 16 or width < 16 or height < 16:
            raise ValueError("生成サイズは 16 の倍数にしてください")
        native = (width, height) == (EXPORT_WIDTH, EXPORT_HEIGHT)
        return {
            "width": width,
            "height": height,
            "export_width": EXPORT_WIDTH,
            "export_height": EXPORT_HEIGHT,
            "native": native,
        }
    if int(vram_mib) >= VRAM_NATIVE_9_16_MIB:
        width, height = GEN_720P
        native = True
    else:
        width, height = GEN_480P
        native = False
    return {
        "width": width,
        "height": height,
        "export_width": EXPORT_WIDTH,
        "export_height": EXPORT_HEIGHT,
        "native": native,
    }


def ffmpeg_vf(geo):
    if geo["native"]:
        return "setsar=1"
    export_w = geo["export_width"]
    export_h = geo["export_height"]
    return (
        f"scale={export_w}:{export_h}:force_original_aspect_ratio=decrease,"
        f"pad={export_w}:{export_h}:(ow-iw)/2:(oh-ih)/2:black,setsar=1"
    )


def ffmpeg_duration_args(frame_count, fps, duration_s):
    produced = frame_count / float(fps)
    if produced > float(duration_s) + 1e-3:
        return ["-t", f"{float(duration_s):.3f}"]
    return []


def ffmpeg_sequence_cmd(sequence_pattern, frame_count, fps, duration_s, geo, dest):
    return [
        "ffmpeg",
        "-y",
        "-framerate",
        str(fps),
        "-i",
        str(sequence_pattern),
        *ffmpeg_duration_args(frame_count, fps, duration_s),
        "-vf",
        ffmpeg_vf(geo),
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        "-movflags",
        "+faststart",
        str(dest),
    ]


def _lerp(samples, t, default):
    if not samples:
        return default
    ordered = sorted(samples, key=lambda item: item[0])
    if t <= ordered[0][0]:
        return ordered[0][1]
    if t >= ordered[-1][0]:
        return ordered[-1][1]
    for (t0, v0), (t1, v1) in zip(ordered, ordered[1:]):
        if t0 <= t <= t1:
            if t1 == t0:
                return v1
            alpha = (t - t0) / (t1 - t0)
            return float(v0) + (float(v1) - float(v0)) * alpha
    return ordered[-1][1]


def contact_allowed(data):
    if "contact" not in data:
        return True
    return bool(data["contact"])


def pose_state_at(data, t):
    facing_samples = []
    paw_samples = []
    jaw_samples = []
    wrist_samples = []
    for kf in data["openpose_keyframes"]:
        ts = float(kf["t"])
        if "cat" in kf:
            pose = _pose_from_name(kf["cat"])
            facing_samples.append((ts, pose["facing"]))
            paw_samples.append((ts, pose["paw"]))
        if "cat_jaw_deg" in kf:
            jaw_samples.append((ts, float(kf["cat_jaw_deg"])))
        if "wrist_to_handle_cm" in kf:
            wrist_samples.append((ts, float(kf["wrist_to_handle_cm"])))
    wrist = None
    if wrist_samples:
        wrist = float(_lerp(wrist_samples, t, wrist_samples[0][1]))
        if not contact_allowed(data):
            wrist = max(wrist, MIN_GAP_CM)
    return {
        "facing": float(_lerp(facing_samples, t, 1.0)),
        "paw": float(_lerp(paw_samples, t, 0.0)),
        "jaw_deg": float(_lerp(jaw_samples, t, 0.0)),
        "wrist_cm": wrist,
    }


def depth_sample(data, t):
    points = [(float(item[0]), str(item[1])) for item in data["depth"]["focus_sweep"]]
    points.sort(key=lambda item: item[0])
    if t <= points[0][0] or t >= points[-1][0]:
        label = points[0][1] if t <= points[0][0] else points[-1][1]
        return label, label, 0.0
    for (t0, left), (t1, right) in zip(points, points[1:]):
        if t0 <= t <= t1:
            alpha = 0.0 if t1 == t0 else (t - t0) / (t1 - t0)
            return left, right, alpha
    label = points[-1][1]
    return label, label, 0.0


def _angle_shift(degrees, span):
    clamped = max(-80.0, min(80.0, float(degrees)))
    return math.tan(math.radians(clamped)) * (float(span) * 0.25)


def handheld_path(frame_count, amp_x, amp_y, seed, irregular):
    count = int(frame_count)
    if count < 1:
        raise ValueError("frame_count")
    rng = random.Random(int(seed))
    if not irregular:
        phase = rng.random() * math.tau
        path = []
        for index in range(count):
            u = index / max(count - 1, 1)
            dx = amp_x * math.sin(2 * math.pi * u + phase)
            dy = amp_y * math.sin(2 * math.pi * u * 1.37 + phase * 0.7)
            path.append((dx, dy))
        return path
    raw = []
    x = 0.0
    y = 0.0
    for _ in range(count):
        x += rng.uniform(-1.0, 1.0)
        y += rng.uniform(-1.0, 1.0)
        raw.append((x, y))
    window = max(3, count // 8)
    smoothed = []
    for index in range(count):
        acc_x = 0.0
        acc_y = 0.0
        n = 0
        start = max(0, index - window)
        end = min(count, index + window + 1)
        for cursor in range(start, end):
            acc_x += raw[cursor][0]
            acc_y += raw[cursor][1]
            n += 1
        smoothed.append((acc_x / n, acc_y / n))
    peak_x = max(abs(point[0]) for point in smoothed) or 1.0
    peak_y = max(abs(point[1]) for point in smoothed) or 1.0
    return [(amp_x * point[0] / peak_x, amp_y * point[1] / peak_y) for point in smoothed]


def camera_delta(camera_vector, frame_index, frame_count, width, height):
    translate = _vec3(camera_vector.get("translate"), "translate")
    rotate = _vec3(camera_vector.get("rotate"), "rotate")
    shake_x = 0.0
    shake_y = 0.0
    shake = camera_vector.get("shake")
    if shake:
        amp = shake.get("amp_px") or [0, 0]
        amp_x = float(amp[0])
        amp_y = float(amp[1]) if len(amp) > 1 else amp_x
        path = handheld_path(
            frame_count,
            amp_x,
            amp_y,
            int(shake.get("seed", 0)),
            bool(shake.get("irregular", False)),
        )
        shake_x, shake_y = path[frame_index]
    scale = 1.0 + (translate[2] / float(height if height else 1))
    scale = max(0.5, min(2.0, scale))
    return {
        "dx": translate[0] + shake_x + _angle_shift(rotate[1], width),
        "dy": translate[1] + shake_y + _angle_shift(rotate[0], height),
        "roll": rotate[2],
        "scale": scale,
    }


def _identity(delta):
    return (
        abs(delta["dx"]) < 1e-6
        and abs(delta["dy"]) < 1e-6
        and abs(delta["roll"]) < 1e-6
        and abs(delta["scale"] - 1.0) < 1e-6
    )


def warp_image(image, delta):
    if _identity(delta):
        return image
    width, height = image.size
    cx = (width - 1) / 2.0
    cy = (height - 1) / 2.0
    scale = delta["scale"]
    theta = math.radians(delta["roll"])
    cos_t = math.cos(theta)
    sin_t = math.sin(theta)
    dx = delta["dx"]
    dy = delta["dy"]
    a = cos_t / scale
    b = sin_t / scale
    d = -sin_t / scale
    e = cos_t / scale
    c = cx - a * (cx + dx) - b * (cy + dy)
    f = cy - d * (cx + dx) - e * (cy + dy)
    fill = 0 if image.mode == "L" else (0, 0, 0)
    return image.transform(
        (width, height),
        Image.Transform.AFFINE,
        (a, b, c, d, e, f),
        resample=Image.Resampling.BICUBIC,
        fillcolor=fill,
    )


def _mix(a, b, facing):
    return (a[0] + (b[0] - a[0]) * facing, a[1] + (b[1] - a[1]) * facing)


def _px(point, width, height):
    return (point[0] * width, point[1] * height)


def _line_width(width):
    return max(2, int(width) // 96)


def cut_mentions_handle(data):
    if any("wrist_to_handle_cm" in kf for kf in data["openpose_keyframes"]):
        return True
    return any(str(item[1]) == "handle" for item in data["depth"]["focus_sweep"])


def draw_pose(state, data, width, height):
    image = Image.new("RGB", (width, height), (0, 0, 0))
    draw = ImageDraw.Draw(image)
    facing = max(0.0, min(1.0, state["facing"]))
    paw = max(0.0, min(1.0, state["paw"]))
    points = {}
    for name in _BACK:
        points[name] = _mix(_BACK[name], _FRONT[name], facing)
    points["lf_paw"] = _mix(_LF_REST, _LF_MOUTH, paw)
    points["rf_paw"] = _RF_REST
    lw = _line_width(width)
    for start, end, color in _LIMBS:
        draw.line([_px(points[start], width, height), _px(points[end], width, height)], fill=color, width=lw)
    head = _px(points["head"], width, height)
    radius = max(2, int(width * 0.03))
    draw.ellipse(
        (head[0] - radius, head[1] - radius, head[0] + radius, head[1] + radius),
        fill=(255, 0, 0),
    )
    if facing > 0.45:
        eye_r = max(1, int(radius * facing * 0.35))
        left_eye = _px((0.34, 0.25), width, height)
        right_eye = _px((0.42, 0.25), width, height)
        for eye, color in ((left_eye, (255, 0, 255)), (right_eye, (170, 0, 255))):
            draw.ellipse(
                (eye[0] - eye_r, eye[1] - eye_r, eye[0] + eye_r, eye[1] + eye_r),
                fill=color,
            )
    if state["jaw_deg"] > 0.5:
        drop = (state["jaw_deg"] / 45.0) * 0.05 * height
        jaw = (head[0], head[1] + radius + drop)
        draw.line([head, jaw], fill=(255, 85, 0), width=lw)
    if state["wrist_cm"] is not None:
        hx = HANDLE_XY[0] * width
        hy = HANDLE_XY[1] * height
        dist = float(state["wrist_cm"]) * CM_OF_FRAME_HEIGHT * height
        wx = hx + WRIST_DIR[0] * dist
        wy = hy + WRIST_DIR[1] * dist
        draw.line([(hx, hy - 0.08 * height), (hx, hy + 0.08 * height)], fill=(0, 255, 255), width=lw)
        elbow = (wx + WRIST_DIR[0] * 0.10 * height, wy + WRIST_DIR[1] * 0.10 * height)
        draw.line([elbow, (wx, wy)], fill=(170, 255, 0), width=lw)
        joint = max(2, lw)
        draw.ellipse((wx - joint, wy - joint, wx + joint, wy + joint), fill=(255, 255, 0))
        if "fingers_visible" in data:
            fingers = int(data["fingers_visible"])
            px, py = -WRIST_DIR[1], WRIST_DIR[0]
            for finger in range(fingers):
                spread = (finger - (fingers - 1) / 2.0) * 0.012 * width
                base = (wx + px * spread * 0.35, wy + py * spread * 0.35)
                tip = (
                    wx - WRIST_DIR[0] * 0.008 * height + px * spread,
                    wy - WRIST_DIR[1] * 0.008 * height + py * spread,
                )
                draw.line([base, tip], fill=(255, 255, 0), width=max(1, lw - 1))
    return image


def _paint_depth(label, data, width, height):
    look = _depth_look(label)
    image = Image.new("L", (width, height), int(look["bg"]))
    draw = ImageDraw.Draw(image)
    scale = float(look["cat_scale"])
    cx = CAT_XY[0] * width
    cy = CAT_XY[1] * height
    rw = 0.14 * width * scale
    rh = 0.24 * height * scale
    draw.ellipse((cx - rw, cy - rh, cx + rw, cy + rh), fill=int(look["cat"]))
    if cut_mentions_handle(data):
        hx = HANDLE_XY[0] * width
        hy = HANDLE_XY[1] * height
        hw = 0.025 * width
        hh = 0.09 * height
        draw.rectangle((hx - hw, hy - hh, hx + hw, hy + hh), fill=int(look["handle"]))
    return image


def draw_depth(label_a, label_b, alpha, data, width, height):
    left = _paint_depth(label_a, data, width, height)
    if label_a == label_b or alpha <= 0.0:
        image = left
    elif alpha >= 1.0:
        image = _paint_depth(label_b, data, width, height)
    else:
        right = _paint_depth(label_b, data, width, height)
        image = Image.blend(left, right, float(alpha))
    if data["depth"].get("sakura_hands") == "defocused":
        top = int(height * 0.68)
        band = image.crop((0, top, width, height)).filter(
            ImageFilter.GaussianBlur(radius=max(2, height // 48))
        )
        image = image.copy()
        image.paste(band, (0, top))
    return image


def composite_controls(pose, depth):
    depth_rgb = depth.convert("RGB")
    mask = pose.convert("L").point(lambda value: 255 if value > 12 else 0)
    return Image.composite(pose, depth_rgb, mask)


def render_control_frame(cut, index, count, width, height):
    if count < 1:
        raise ValueError("count")
    if index < 0 or index >= count:
        raise IndexError(index)
    t = frame_source_time(cut, index, count)
    data = cut["data"]
    state = pose_state_at(data, t)
    label_a, label_b, alpha = depth_sample(data, t)
    pose = draw_pose(state, data, width, height)
    depth = draw_depth(label_a, label_b, alpha, data, width, height)
    delta = camera_delta(cut["camera_vector"], index, count, width, height)
    pose_w = warp_image(pose, delta)
    depth_w = warp_image(depth, delta)
    return {
        "time": t,
        "pose_state": state,
        "depth": (label_a, label_b, alpha),
        "camera": delta,
        "pose_raw": pose,
        "depth_raw": depth,
        "pose": pose_w,
        "depth_out": depth_w,
        "composite": composite_controls(pose_w, depth_w),
    }


def render_cut(cut, dest_dir, width, height):
    dest = Path(dest_dir)
    dest.mkdir(parents=True, exist_ok=True)
    for old in dest.glob("frame_*.png"):
        old.unlink()
    count = wan_frame_count(cut["duration_s"], cut["fps"])
    metas = []
    for index in range(count):
        frame = render_control_frame(cut, index, count, width, height)
        frame["composite"].save(dest / f"frame_{index:04d}.png")
        metas.append(
            {
                "time": frame["time"],
                "camera": frame["camera"],
                "pose_state": frame["pose_state"],
                "depth": frame["depth"],
            }
        )
    return {"directory": str(dest), "count": count, "metas": metas, "width": width, "height": height}


def letterbox_face(image, width, height):
    if image.mode != "RGB":
        image = image.convert("RGB")
    canvas = Image.new("RGB", (int(width), int(height)), FACE_PAD)
    fitted = image.copy()
    max_w = max(1, int(width * 0.72))
    max_h = max(1, int(height * 0.42))
    fitted.thumbnail((max_w, max_h), Image.Resampling.LANCZOS)
    x = (int(width) - fitted.width) // 2
    y = max(0, int(height * 0.08))
    canvas.paste(fitted, (x, y))
    return canvas


def build_prompts(data):
    lines = [
        "vertical 9:16 shot",
        "the person's face matches the reference image",
        "follow the OpenPose control, the depth control, and the camera move",
        f"duration {float(data['duration_s'])} seconds",
    ]
    if data.get("sakura") == "static":
        lines.append("the cat stays in place except for its jaw")
    for kf in data["openpose_keyframes"]:
        t = kf["t"]
        if "cat" in kf:
            lines.append(f"at {t}s cat pose {str(kf['cat']).replace('_', ' ')}")
        if "cat_jaw_deg" in kf:
            lines.append(f"at {t}s cat jaw {kf['cat_jaw_deg']} degrees")
        if "wrist_to_handle_cm" in kf:
            lines.append(f"at {t}s wrist is {kf['wrist_to_handle_cm']} cm from the handle")
    if data.get("contact") is False:
        lines.append("the hand does not touch the handle")
    if data.get("hard_cut_before_touch") is True:
        lines.append("cut before the hand touches")
    if "fingers_visible" in data:
        lines.append(f"{int(data['fingers_visible'])} fingers are visible")
    for cap in data.get("caption") or []:
        lines.append(f"spoken line, not written on screen: {cap['text']}")
    positive = ". ".join(lines)
    negative_parts = [
        "low quality",
        "blurry",
        "watermark",
        "subtitles",
        "captions",
        "on-screen text",
        "letters",
        "extra limbs",
        "deformed hands",
        "frozen video",
    ]
    if data.get("contact") is False:
        negative_parts.append("hand touching the handle")
        negative_parts.append("grip")
    return positive, ", ".join(negative_parts)


def summarize_cut(cut, frame_count):
    return (
        f"{cut['cut']}: duration_s={cut['duration_s']} fps={cut['fps']} "
        f"frames={frame_count} source={list(cut['source_seconds'])} "
        f"openpose={len(cut['openpose_keyframes'])} "
        f"depth={len(cut['depth']['focus_sweep'])}"
    )


def sampler_seed(cut):
    shake = cut["camera_vector"].get("shake") or {}
    return 10000 + int(shake.get("seed", 0))


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

NODE_SOURCE = '"""Load a directory of frame_*.png as a ComfyUI IMAGE batch.\n\nThe H3 VACE notebook writes this file into ComfyUI/custom_nodes before startup.\nIt does not download model weights.\n"""\n\nimport glob\nimport os\n\nimport numpy as np\nimport torch\nfrom PIL import Image\n\n\nclass H3LoadFrameDir:\n    @classmethod\n    def INPUT_TYPES(cls):\n        return {\n            "required": {\n                "directory": ("STRING", {"default": ""}),\n            }\n        }\n\n    RETURN_TYPES = ("IMAGE",)\n    FUNCTION = "load"\n    CATEGORY = "h3"\n\n    def load(self, directory):\n        pattern = os.path.join(directory, "frame_*.png")\n        paths = sorted(glob.glob(pattern))\n        if not paths:\n            raise FileNotFoundError(f"制御フレームがありません: {directory}")\n        frames = []\n        for path in paths:\n            image = Image.open(path).convert("RGB")\n            array = np.asarray(image).astype("float32") / 255.0\n            frames.append(torch.from_numpy(array))\n        batch = torch.stack(frames, dim=0)\n        return (batch,)\n\n\nNODE_CLASS_MAPPINGS = {"H3LoadFrameDir": H3LoadFrameDir}\nNODE_DISPLAY_NAME_MAPPINGS = {"H3LoadFrameDir": "H3 Load Frame Directory"}\n'

from google.colab import drive

if not globals().get("GPU_OK"):
    raise RuntimeError(
        "GPU チェックを通過していません。最初のセルから順に実行してください。"
        "A100 未満の GPU ではここで止めます。"
    )

print("Google Drive をマウントします。")
drive.mount("/content/drive")

DRIVE_BASE = "/content/drive/MyDrive/ComfyUI_models"
WORK_ROOT = "/content/drive/MyDrive/ComfyUI_vace_h3"
FACE_DIR = WORK_ROOT + "/input"
CONTROL_DIR = WORK_ROOT + "/input/controls"
OUTPUT_ROOT = WORK_ROOT + "/output"
COMFY_DIR = "/content/ComfyUI"

for folder in (
    f"{DRIVE_BASE}/diffusion_models",
    f"{DRIVE_BASE}/text_encoders",
    f"{DRIVE_BASE}/clip",
    f"{DRIVE_BASE}/vae",
    FACE_DIR,
    CONTROL_DIR,
    OUTPUT_ROOT,
):
    os.makedirs(folder, exist_ok=True)

clone_env = os.environ.copy()
clone_env["GIT_TERMINAL_PROMPT"] = "0"
if not os.path.isdir(COMFY_DIR + "/.git"):
    print("ComfyUI を取得します。")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", COMFY_DIR],
        check=True,
        env=clone_env,
    )
else:
    print("ComfyUI を更新します。")
    updated = subprocess.run(["git", "-C", COMFY_DIR, "pull", "--ff-only"], env=clone_env)
    if updated.returncode != 0:
        print("ComfyUI の更新をスキップし、既存の取得物で続行します。")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", COMFY_DIR + "/requirements.txt"],
    check=True,
)
subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)

node_dir = Path(COMFY_DIR) / "custom_nodes"
node_dir.mkdir(parents=True, exist_ok=True)
node_path = node_dir / "h3_load_frames.py"
node_path.write_text(NODE_SOURCE, encoding="utf-8")
print(f"フレーム読み込みノード: {node_path}")
print(f"モデル置き場: {DRIVE_BASE}")
print(f"顔画像: {FACE_DIR}/face_ref.png （jpg / webp も可）")
print(f"制御 JSON: {CONTROL_DIR}")
print(f"出力: {OUTPUT_ROOT}")
print("セットアップセル完了")


In [ ]:
import os
import subprocess
from pathlib import Path

if not globals().get("GPU_OK"):
    raise RuntimeError("GPU チェックを通過していません。最初のセルから実行してください。")
if not os.path.isdir("/content/ComfyUI"):
    raise RuntimeError("セットアップセルを先に実行してください。")

DRIVE_BASE = "/content/drive/MyDrive/ComfyUI_models"
HF = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main"
UNET_FILE = "wan2.1_vace_14B_fp16.safetensors"
TE_FILE = "umt5_xxl_fp8_e4m3fn_scaled.safetensors"
VAE_FILE = "wan_2.1_vae.safetensors"

FILES = [
    {
        "name": UNET_FILE,
        "url": f"{HF}/split_files/diffusion_models/{UNET_FILE}",
        "dest": f"{DRIVE_BASE}/diffusion_models/{UNET_FILE}",
        "alts": [],
        "min_bytes": 32_000_000_000,
    },
    {
        "name": TE_FILE,
        "url": f"{HF}/split_files/text_encoders/{TE_FILE}",
        "dest": f"{DRIVE_BASE}/text_encoders/{TE_FILE}",
        "alts": [f"{DRIVE_BASE}/clip/{TE_FILE}"],
        "min_bytes": 6_400_000_000,
    },
    {
        "name": VAE_FILE,
        "url": f"{HF}/split_files/vae/{VAE_FILE}",
        "dest": f"{DRIVE_BASE}/vae/{VAE_FILE}",
        "alts": [],
        "min_bytes": 240_000_000,
    },
]


def file_ok(path, min_bytes):
    return os.path.isfile(path) and os.path.getsize(path) >= min_bytes


resolved = {}
to_download = []
for item in FILES:
    found = None
    for path in [item["dest"], *item["alts"]]:
        if file_ok(path, item["min_bytes"]):
            found = path
            break
    if found:
        print(f"スキップ（Drive に既存 {os.path.getsize(found) / 1e9:.1f} GB）: {found}")
        resolved[item["name"]] = found
    else:
        to_download.append(item)

if to_download:
    stat = os.statvfs("/content/drive/MyDrive")
    free = stat.f_bavail * stat.f_frsize
    need = sum(item["min_bytes"] for item in to_download)
    print(f"Drive の空き目安: {free / 1e9:.1f} GB / ダウンロード目安: {need / 1e9:.1f} GB")
    if free < need:
        raise RuntimeError(
            "Drive の空きが足りない目安です。モデルは約 39GB あります。"
            "空きを作ってから、このセルを再実行してください。"
        )

for item in to_download:
    dest = item["dest"]
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    if os.path.isfile(dest) and not file_ok(dest, item["min_bytes"]):
        print(f"不完全なファイルを消して取り直します: {dest}")
        os.remove(dest)
    print(f"ダウンロード: {item['name']}")
    result = subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", dest, item["url"]])
    if result.returncode != 0 or not file_ok(dest, item["min_bytes"]):
        raise RuntimeError(
            f"ダウンロードに失敗しました: {item['name']}。このセルを再実行すると続きから取得します。"
        )
    resolved[item["name"]] = dest
    print(f"完了: {os.path.getsize(dest) / 1e9:.1f} GB")


def rel_folder(path):
    return os.path.relpath(os.path.dirname(path), DRIVE_BASE).replace("\\", "/")


yaml_path = "/content/ComfyUI/extra_model_paths.yaml"
unet_folder = rel_folder(resolved[UNET_FILE])
yaml_text = (
    "h3_vace:\n"
    f"    base_path: {DRIVE_BASE}\n"
    f"    diffusion_models: {unet_folder}/\n"
    f"    unet: {unet_folder}/\n"
    f"    text_encoders: {rel_folder(resolved[TE_FILE])}/\n"
    f"    vae: {rel_folder(resolved[VAE_FILE])}/\n"
    "    clip: clip/\n"
)
Path(yaml_path).write_text(yaml_text, encoding="utf-8")
MODEL_UNET = resolved[UNET_FILE]
MODEL_TE = resolved[TE_FILE]
MODEL_VAE = resolved[VAE_FILE]
print(yaml_path)
print("モデルセル完了。ComfyUI は次のセルで起動します。")


In [ ]:
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

if not globals().get("GPU_OK"):
    raise RuntimeError("GPU チェックを通過していません。最初のセルから実行してください。")
if not os.path.isfile("/content/ComfyUI/extra_model_paths.yaml"):
    raise RuntimeError("モデルのセルが未完了です。extra_model_paths.yaml がありません。")
if not os.path.isfile("/content/ComfyUI/custom_nodes/h3_load_frames.py"):
    raise RuntimeError("セットアップセルが未完了です。フレーム読み込みノードがありません。")

LOG_PATH = "/content/comfyui_vace.log"


def comfy_is_up():
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=3) as resp:
            return resp.status == 200
    except Exception:
        return False


if comfy_is_up():
    print("ComfyUI は既に起動しています。")
else:
    COMFY_LOG = open(LOG_PATH, "w", encoding="utf-8")
    COMFY_PROC = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188"],
        cwd="/content/ComfyUI",
        stdout=COMFY_LOG,
        stderr=subprocess.STDOUT,
    )
    print("ComfyUI を起動しています。")
    ready = False
    for _ in range(150):
        if COMFY_PROC.poll() is not None:
            break
        if comfy_is_up():
            ready = True
            break
        time.sleep(2)
        print(".", end="", flush=True)
    print()
    if not ready:
        COMFY_LOG.flush()
        text = Path(LOG_PATH).read_text(encoding="utf-8", errors="replace")
        print(text[-2000:])
        raise RuntimeError("ComfyUI が起動しませんでした。上のログを確認してください。")
    print("ComfyUI 起動完了")


In [ ]:
import json
import os
import shutil
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

from PIL import Image

if not globals().get("GPU_OK"):
    raise RuntimeError("GPU チェックを通過していません。最初のセルから実行してください。")
for _name in (
    "list_cuts",
    "render_cut",
    "generation_geometry",
    "build_prompts",
    "ffmpeg_vf",
    "ffmpeg_duration_args",
    "ffmpeg_sequence_cmd",
    "copy_control_json",
    "find_face_reference",
    "safe_cut_id",
    "summarize_cut",
    "raw_frame_count",
    "letterbox_face",
    "sampler_seed",
    "EXPORT_WIDTH",
    "EXPORT_HEIGHT",
):
    if _name not in globals():
        raise RuntimeError("制御 JSON のセルが未実行です。上から順に実行してください。")
if "FACE_DIR" not in globals() or "CONTROL_DIR" not in globals() or "OUTPUT_ROOT" not in globals():
    raise RuntimeError("セットアップセルが未実行です。")
if not os.path.isdir("/content/ComfyUI"):
    raise RuntimeError("ComfyUI がありません。セットアップセルを実行してください。")

UNET_FILE = "wan2.1_vace_14B_fp16.safetensors"
TE_FILE = "umt5_xxl_fp8_e4m3fn_scaled.safetensors"
VAE_FILE = "wan_2.1_vae.safetensors"
# 蒸留 LoRA は同梱しない。Wan 14B を uni_pc / shift 8 で回すときの通常値。
STEPS = 20
CFG = 5.0
SHIFT = 8.0
POLL_SECONDS = 10
CUT_TIMEOUT_SECONDS = 3 * 60 * 60
# None のときは VRAM で決める。A100 80GB 以上と同じ 720x1280 にするなら (720, 1280)。
GEN_SIZE_OVERRIDE = None

GENERATION_DONE = False
OUTPUT_MP4S = []
OUTPUT_JSONS = []


def comfy_call(path, payload=None, timeout=120):
    url = "http://127.0.0.1:8188" + path
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    headers = {"Content-Type": "application/json"} if data is not None else {}
    req = urllib.request.Request(url, data=data, headers=headers)
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return json.loads(resp.read().decode("utf-8"))
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"ComfyUI {path} -> {exc.code}: {body}") from exc


def pack_inputs(info, class_type, values):
    spec = info[class_type]["input"]
    required = set(spec.get("required", {}))
    optional = set(spec.get("optional", {}))
    allowed = required | optional
    missing = sorted(key for key in required if key not in values)
    if missing:
        raise RuntimeError(f"{class_type} に足りない入力: {missing}")
    return {key: value for key, value in values.items() if key in allowed}


def build_prompt(info, frame_dir, length, width, height, positive, negative, seed, prefix):
    vace_spec = info["WanVaceToVideo"]["input"]
    vace_allowed = set(vace_spec.get("required", {})) | set(vace_spec.get("optional", {}))
    for key in ("control_video", "reference_image"):
        if key not in vace_allowed:
            raise RuntimeError(f"WanVaceToVideo に {key} がありません。ComfyUI を更新してください。")
    return {
        "unet": {
            "class_type": "UNETLoader",
            "inputs": pack_inputs(info, "UNETLoader", {
                "unet_name": UNET_FILE,
                "weight_dtype": "default",
            }),
        },
        "clip": {
            "class_type": "CLIPLoader",
            "inputs": pack_inputs(info, "CLIPLoader", {
                "clip_name": TE_FILE,
                "type": "wan",
                "device": "default",
            }),
        },
        "vae": {
            "class_type": "VAELoader",
            "inputs": pack_inputs(info, "VAELoader", {"vae_name": VAE_FILE}),
        },
        "pos": {
            "class_type": "CLIPTextEncode",
            "inputs": pack_inputs(info, "CLIPTextEncode", {"text": positive, "clip": ["clip", 0]}),
        },
        "neg": {
            "class_type": "CLIPTextEncode",
            "inputs": pack_inputs(info, "CLIPTextEncode", {"text": negative, "clip": ["clip", 0]}),
        },
        "face": {
            "class_type": "LoadImage",
            "inputs": pack_inputs(info, "LoadImage", {"image": "face_ref.png"}),
        },
        "frames": {
            "class_type": "H3LoadFrameDir",
            "inputs": pack_inputs(info, "H3LoadFrameDir", {"directory": frame_dir}),
        },
        "vace": {
            "class_type": "WanVaceToVideo",
            "inputs": pack_inputs(info, "WanVaceToVideo", {
                "positive": ["pos", 0],
                "negative": ["neg", 0],
                "vae": ["vae", 0],
                "width": width,
                "height": height,
                "length": length,
                "batch_size": 1,
                "strength": 1.0,
                "control_video": ["frames", 0],
                "reference_image": ["face", 0],
            }),
        },
        "shift": {
            "class_type": "ModelSamplingSD3",
            "inputs": pack_inputs(info, "ModelSamplingSD3", {"model": ["unet", 0], "shift": SHIFT}),
        },
        "sample": {
            "class_type": "KSampler",
            "inputs": pack_inputs(info, "KSampler", {
                "model": ["shift", 0],
                "positive": ["vace", 0],
                "negative": ["vace", 1],
                "latent_image": ["vace", 2],
                "seed": seed,
                "steps": STEPS,
                "cfg": CFG,
                "sampler_name": "uni_pc",
                "scheduler": "simple",
                "denoise": 1.0,
                "control_after_generate": "fixed",
            }),
        },
        "trim": {
            "class_type": "TrimVideoLatent",
            "inputs": pack_inputs(info, "TrimVideoLatent", {
                "samples": ["sample", 0],
                "trim_amount": ["vace", 3],
            }),
        },
        "decode": {
            "class_type": "VAEDecodeTiled",
            "inputs": pack_inputs(info, "VAEDecodeTiled", {
                "samples": ["trim", 0],
                "vae": ["vae", 0],
                "tile_size": 512,
                "overlap": 64,
                "temporal_size": 64,
                "temporal_overlap": 4,
            }),
        },
        "save": {
            "class_type": "SaveImage",
            "inputs": pack_inputs(info, "SaveImage", {
                "images": ["decode", 0],
                "filename_prefix": prefix,
            }),
        },
    }


def wait_for_cut(prompt_id, cut_id):
    started = time.time()
    announced = -1
    while True:
        elapsed = time.time() - started
        if elapsed > CUT_TIMEOUT_SECONDS:
            raise RuntimeError(f"{cut_id} が制限時間を超えました。ランタイムは消しません。")
        hist = comfy_call(f"/history/{prompt_id}")
        if prompt_id in hist:
            item = hist[prompt_id]
            status = item.get("status") or {}
            if status.get("status_str") == "error":
                detail = json.dumps(status.get("messages"), ensure_ascii=False)[:4000]
                lowered = detail.lower()
                if "out of memory" in lowered or "allocation" in lowered:
                    print(
                        "VRAM 不足です。GEN_SIZE_OVERRIDE を None のままにするか、"
                        "より大きい GPU で (720, 1280) を指定してください。"
                    )
                raise RuntimeError(f"{cut_id} の生成に失敗しました: {detail}")
            if status.get("completed") or item.get("outputs"):
                return item
        minute = int(elapsed // 60)
        if minute != announced:
            print(f"{cut_id}: 生成中 {minute} 分")
            announced = minute
        time.sleep(POLL_SECONDS)


def export_mp4(image_paths, fps, duration_s, geo, dest):
    seq = Path("/content/vace_work/seq")
    if seq.exists():
        shutil.rmtree(seq)
    seq.mkdir(parents=True)
    for index, path in enumerate(image_paths):
        shutil.copyfile(path, seq / f"frame_{index:04d}.png")
    cmd = ffmpeg_sequence_cmd(
        seq / "frame_%04d.png",
        len(image_paths),
        fps,
        duration_s,
        geo,
        dest,
    )
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg に失敗しました: {result.stderr[-2000:]}")
    probe = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",
            "-select_streams",
            "v:0",
            "-show_entries",
            "stream=width,height",
            "-of",
            "csv=p=0:s=x",
            str(dest),
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    got = probe.stdout.strip()
    expect = f"{EXPORT_WIDTH}x{EXPORT_HEIGHT}"
    if got != expect:
        raise RuntimeError(f"書き出しサイズが {got} です。{expect} が必要です。")


if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    raise RuntimeError("ffmpeg がありません。セットアップセルを再実行してください。")

try:
    comfy_call("/system_stats", timeout=5)
except Exception as exc:
    raise RuntimeError("ComfyUI が起動していません。起動セルを実行してください。") from exc

info = comfy_call("/object_info")
for node_name in (
    "UNETLoader",
    "CLIPLoader",
    "VAELoader",
    "CLIPTextEncode",
    "LoadImage",
    "WanVaceToVideo",
    "ModelSamplingSD3",
    "KSampler",
    "TrimVideoLatent",
    "VAEDecodeTiled",
    "SaveImage",
    "H3LoadFrameDir",
):
    if node_name not in info:
        raise RuntimeError(f"ノードがありません: {node_name}。セットアップのあとランタイムを入れ直してください。")

geo = generation_geometry(_GPU_VRAM, GEN_SIZE_OVERRIDE)
print(
    f"生成 {geo['width']}x{geo['height']} / 書き出し {geo['export_width']}x{geo['export_height']}"
    f"（9:16{'、そのまま' if geo['native'] else '、480p をパッド'}）"
)
face_path = find_face_reference(FACE_DIR)
boxed = letterbox_face(Image.open(face_path), geo["width"], geo["height"])
os.makedirs("/content/ComfyUI/input", exist_ok=True)
boxed.save("/content/ComfyUI/input/face_ref.png")
print(f"顔参照: {face_path}")

cuts = list_cuts(CONTROL_DIR)
run_id = time.strftime("%Y%m%d-%H%M%S")
run_dir = Path(OUTPUT_ROOT) / run_id
run_dir.mkdir(parents=True, exist_ok=True)
print(f"出力フォルダ: {run_dir}")

for cut in cuts:
    cut_id = safe_cut_id(cut["cut"])
    count = render_cut(
        cut,
        f"/content/vace_work/{cut_id}/frames",
        geo["width"],
        geo["height"],
    )["count"]
    raw = raw_frame_count(cut["duration_s"], cut["fps"])
    print(summarize_cut(cut, count))
    if count != raw:
        print(f"注意: {raw} フレームを VACE の制約で {count} フレームに揃えます（4n+1、最大 81）。")
    positive, negative = build_prompts(cut["data"])
    print(positive)
    prefix = f"h3_{run_id}_{cut_id}"
    out_root = Path("/content/ComfyUI/output")
    out_root.mkdir(parents=True, exist_ok=True)
    for old in out_root.glob(prefix + "*"):
        if old.is_file():
            old.unlink()
    prompt = build_prompt(
        info,
        f"/content/vace_work/{cut_id}/frames",
        count,
        geo["width"],
        geo["height"],
        positive,
        negative,
        sampler_seed(cut),
        prefix,
    )
    queued = comfy_call("/prompt", {"prompt": prompt})
    if queued.get("node_errors") or "prompt_id" not in queued:
        raise RuntimeError(json.dumps(queued, ensure_ascii=False)[:4000])
    item = wait_for_cut(queued["prompt_id"], cut_id)
    images = []
    for node_out in item.get("outputs", {}).values():
        images.extend(node_out.get("images") or [])
    paths = []
    for image in images:
        root = "/content/ComfyUI/temp" if image.get("type") == "temp" else "/content/ComfyUI/output"
        sub = image.get("subfolder") or ""
        paths.append(os.path.join(root, sub, image["filename"]))
    paths = [path for path in paths if os.path.isfile(path)]
    paths.sort()
    if len(paths) != count:
        raise RuntimeError(f"{cut_id}: フレーム数が {len(paths)} でした。{count} を期待しています。")
    local_mp4 = Path(f"/content/vace_work/{cut_id}.mp4")
    export_mp4(paths, cut["fps"], cut["duration_s"], geo, local_mp4)
    dest_mp4 = run_dir / f"{cut_id}.mp4"
    shutil.copyfile(local_mp4, dest_mp4)
    dest_json = copy_control_json(cut, run_dir)
    if dest_mp4.stat().st_size <= 0 or dest_json.stat().st_size <= 0:
        raise RuntimeError(f"{cut_id}: 出力が空です")
    OUTPUT_MP4S.append(str(dest_mp4))
    OUTPUT_JSONS.append(str(dest_json))
    print(f"書き出し: {dest_mp4}")
    print(f"制御 JSON: {dest_json}")
    try:
        comfy_call("/free", {"unload_models": False, "free_memory": True})
    except Exception as exc:
        print(f"メモリ解放をスキップ: {exc}")

if len(OUTPUT_MP4S) != len(cuts) or len(OUTPUT_JSONS) != len(cuts):
    raise RuntimeError("出力本数がカット数と一致しません。")
GENERATION_DONE = True
print("全カットの mp4 と制御 JSON を Drive に書きました。")
print("次のセルでランタイムを削除します。")


In [ ]:
import os

from google.colab import runtime

if not globals().get("GENERATION_DONE"):
    raise RuntimeError(
        "生成が終わっていないので、ランタイムは切断しません。失敗したセルの出力を確認してください。"
    )
paths = list(globals().get("OUTPUT_MP4S") or []) + list(globals().get("OUTPUT_JSONS") or [])
if not paths:
    raise RuntimeError("出力パスが空なので、ランタイムは切断しません。")
for path in paths:
    if not os.path.isfile(path) or os.path.getsize(path) == 0:
        raise RuntimeError(f"出力が無いので、ランタイムは切断しません: {path}")
print("mp4 と制御 JSON を確認しました。ランタイムを切断して削除します。")
runtime.unassign()
